In [1]:
import fitz
from langchain_core.documents import Document
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import numpy as np
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.embeddings import Embeddings
import os
import base64
import io
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

c:\Users\ksasa\OneDrive\Desktop\rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\ksasa\AppData\Local\Temp\ipykernel_12240\127754628.py:16: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
from dotenv import load_dotenv
load_dotenv() 

clip_model =CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 15372.66it/s]


CLIPModel(
  (text_model): CLIPTextModel(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1

In [3]:
def embed_img(image):
    inputs = clip_processor(
        images=image,
        return_tensors="pt"
    )

    with torch.no_grad():
        image_outputs = clip_model.vision_model(**inputs)

        # Get the pooled image representation
        pooled_output = image_outputs.pooler_output

        # Project into CLIP embedding space
        features = clip_model.visual_projection(pooled_output)

        # Normalize
        features = features / features.norm(dim=-1, keepdim=True)

    return features.squeeze().cpu().numpy()

def embed_txt(text):
    inputs = clip_processor(
        text=[text],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77
    )

    with torch.no_grad():
        text_outputs = clip_model.text_model(**inputs)

        # Get the pooled text representation
        pooled_output = text_outputs.pooler_output

        # Project it into CLIP's embedding space
        features = clip_model.text_projection(pooled_output)

        # Normalize the embedding
        features = features / features.norm(dim=-1, keepdim=True)

    return features.squeeze().cpu().numpy()

In [4]:
pdf_path = "multimodal_sample.pdf"
doc = fitz.open(pdf_path)

all_docs = []
all_embeddings = []
image_data_store = {}

splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 100)

In [5]:
for i, page in enumerate(doc):
    text = page.get_text()

    if text.strip():
        temp_doc = Document(
            page_content=text,
            metadata={
                "page": i + 1,
                "type": "text"
            }
        )

        text_chunks = splitter.split_documents([temp_doc])

        for chunk in text_chunks:
            embedding = embed_txt(chunk.page_content)
            all_embeddings.append(embedding)
            all_docs.append(chunk)

    for img_index, img in enumerate(page.get_images(full=True)):
        try:
            xref = img[0]

            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]

            pil_image = Image.open(
                io.BytesIO(image_bytes)
            ).convert("RGB")

            image_id = f"page_{i + 1}_img_{img_index}"

            buffered = io.BytesIO()
            pil_image.save(buffered, format="PNG")

            # Encode raw PNG bytes → Base64
            img_base64 = base64.b64encode(
                buffered.getvalue()
            ).decode("utf-8")

            image_data_store[image_id] = img_base64

            embedding = embed_img(pil_image)
            all_embeddings.append(embedding)

            img_doc = Document(
                page_content=f"[Image:{image_id}]",
                metadata={
                    "page": i + 1,
                    "type": "image",
                    "image_id": image_id
                }
            )

            all_docs.append(img_doc)

        except Exception as e:
            print(
                f"Error processing image {img_index} "
                f"on page {i + 1}: {e}"
            )
            continue

# Close the PDF AFTER all pages have been processed
doc.close()

In [6]:
all_embeddings

[array([-2.67243781e-03,  1.28300041e-02, -5.18313907e-02,  4.14879583e-02,
        -2.33942047e-02, -7.55864149e-03, -3.67659293e-02,  1.19710729e-01,
         8.52080807e-02,  2.05425359e-03, -1.11535070e-02, -1.29592502e-02,
         5.25014624e-02, -3.65393190e-03,  4.76078577e-02,  1.58372652e-02,
         2.03388128e-02,  4.35362011e-02, -3.29168560e-03,  2.03181971e-02,
         1.88025821e-03, -4.23493683e-02,  5.44103002e-03,  3.70935872e-02,
        -1.65622979e-02,  6.48646569e-03, -4.78012264e-02,  8.67482368e-03,
         5.88859729e-02, -3.21394205e-02,  4.32439931e-02,  9.65301599e-03,
        -4.47921269e-03, -1.94858089e-02, -3.63502726e-02, -1.23471599e-02,
        -2.17929129e-02, -1.99016277e-02,  8.09619576e-02, -3.32986787e-02,
        -2.38901358e-02, -3.96138877e-02, -1.27279665e-02,  3.50380950e-02,
        -2.52217147e-02,  2.00030068e-03,  1.49660315e-02, -2.31976435e-02,
        -6.86791539e-02, -5.25792770e-04, -2.22545601e-02, -1.04104010e-02,
        -1.9

In [7]:
all_docs

[Document(metadata={'page': 1, 'type': 'text'}, page_content='Annual Revenue Overview\nThis document summarizes the revenue trends across Q1, Q2, and Q3. As illustrated in the chart\nbelow, revenue grew steadily with the highest growth recorded in Q3.\nQ1 showed a moderate increase in revenue as new product lines were introduced. Q2 outperformed\nQ1 due to marketing campaigns. Q3 had exponential growth due to global expansion.'),
 Document(metadata={'page': 1, 'type': 'image', 'image_id': 'page_1_img_0'}, page_content='[Image:page_1_img_0]')]

In [8]:
class CLIPEmbeddings(Embeddings):
    def embed_documents(self, texts):
        return [embed_txt(text).tolist() for text in texts]

    def embed_query(self, text):
        return embed_txt(text).tolist()

In [9]:
embedding_array = np.array(all_embeddings, dtype="float32")

vector_store = FAISS.from_embeddings(
    text_embeddings=[
        (doc.page_content, emb.tolist())
        for doc, emb in zip(all_docs, embedding_array)
    ],
    embedding=CLIPEmbeddings(),
    metadatas=[doc.metadata for doc in all_docs]
)

vector_store

In [10]:
llm = init_chat_model(
    "openai/gpt-4o-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)
llm

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.5.5', 'langchain': '1.3.15', 'langchain-openai': '1.5.1'}}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001D448842CD0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001D4492473D0>, root_client=<openai.OpenAI object at 0x000001D448843550>, root_async_client=<openai.AsyncOpenAI object at 0x000001D449246F50>, model_name='openai/gpt-4o-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://openrouter.ai/api/v1', stream_chunk_timeout=120.0)

In [11]:
def retrieve_multimodel(query, k=5):
    query_embedding = embed_txt(query)

    results = vector_store.similarity_search_by_vector(
        embedding = query_embedding,
        k=k
    )

    return results

In [12]:
def create_multimodal_message(query, retrieved_docs):
    content = []

    content.append({
        "type": "text",
        "text": f"Question: {query}\n\nContent:\n"
    })

    text_docs = [
        doc for doc in retrieved_docs
        if doc.metadata.get("type") == "text"
    ]

    image_docs = [
        doc for doc in retrieved_docs
        if doc.metadata.get("type") == "image"
    ]

    # Add text context
    if text_docs:
        text_context = "\n\n".join([
            f"[Page {doc.metadata['page']}]: {doc.page_content}"
            for doc in text_docs
        ])

        content.append({
            "type": "text",
            "text": f"Text excerpts:\n{text_context}\n"
        })

    # Add images
    for doc in image_docs:
        image_id = doc.metadata.get("image_id")

        if image_id and image_id in image_data_store:
            content.append({
                "type": "text",
                "text": f"\n[Image from page {doc.metadata['page']}]:\n"
            })

            content.append({
                "type": "image_url",
                "image_url": {
                    "url": (
                        f"data:image/png;base64,"
                        f"{image_data_store[image_id]}"
                    )
                }
            })

    content.append({
        "type": "text",
        "text": (
            "\n\nPlease answer the question based on "
            "the provided text and images."
        )
    })

    return HumanMessage(content=content)

In [13]:
def multimodal_pdf_rag_pipeline(query):
    """Main pipeline for multimodal RAG."""
    context_docs = retrieve_multimodel(query, k=5)
    message = create_multimodal_message(query, context_docs)
    response = llm.invoke([message])
    
    print(f"\nRetrieved {len(context_docs)} documents:")
    for doc in context_docs:
        doc_type = doc.metadata.get("type", "unknown")
        page = doc.metadata.get("page", "?")
        if doc_type == "text":
            preview = doc.page_content[:100] + "..." if len(doc.page_content) > 100 else doc.page_content
            print(f"  - Text from page {page}: {preview}")
        else:
            print(f"  - Image from page {page}")
    print("\n")
    
    return response.content

In [15]:
if __name__ == "__main__":
    queries = [
        "What does the document talk about?",
        "Summarize the main findings from the document",
        "What visual elements are present in the document?"
    ]

    for query in queries:
        print (f"\nQuery:{query}")
        print("-" * 50)
        answer = multimodal_pdf_rag_pipeline(query)
        print(f"\nAnswer:{answer}")
        print("=" * 70)


Query:What does the document talk about?
--------------------------------------------------

Retrieved 2 documents:
  - Text from page 1: Annual Revenue Overview
This document summarizes the revenue trends across Q1, Q2, and Q3. As illust...
  - Image from page 1



Answer:The document discusses the annual revenue trends for a company over three quarters (Q1, Q2, and Q3). It highlights that revenue growth was steady, with Q3 showing the highest increase due to global expansion. Q1 saw a moderate increase due to new product lines, while Q2 outperformed Q1 thanks to marketing campaigns. The accompanying chart visually represents this revenue growth across the quarters, showing blue for Q1, green for Q2, and red for Q3.

Query:Summarize the main findings from the document
--------------------------------------------------

Retrieved 2 documents:
  - Text from page 1: Annual Revenue Overview
This document summarizes the revenue trends across Q1, Q2, and Q3. As illust...
  - Image from pag